In [1]:
import os, glob, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

os.makedirs("results/figures", exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

# часове розбиття (configs/ieee_cis_split.yaml)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
for n, m in [("train", m_tr), ("val", m_va), ("test", m_te)]:
    print(f"{n}: {m.sum()} транзакцій, фрод {df.isFraud[m].sum()} ({df.isFraud[m].mean():.2%})")

# ознаки
X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
y = df.isFraud.values
Xtr, ytr, Xva, yva, Xte, yte = X[m_tr], y[m_tr], X[m_va], y[m_va], X[m_te], y[m_te]
spw = (ytr == 0).sum() / (ytr == 1).sum()

def metrics(y_true, p):
    k = int(y_true.sum()); order = np.argsort(-p)
    top1 = order[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p),
            "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[order[:k]].sum() / k,
            "precision@1%": y_true[top1].mean(),
            "base_rate": y_true.mean()}

# GPU-налаштування залежно від версії XGBoost
major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")
print("xgboost", xgb.__version__, DEV)

def run_xgb(seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                      eval_metric="aucpr", early_stopping_rounds=50,
                      random_state=seed, **DEV)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    r = metrics(yte, m.predict_proba(Xte)[:, 1])
    r.update(seed=seed, best_iter=m.best_iteration, val_aucpr=m.best_score)
    return r

train: 431125 транзакцій, фрод 15150 (3.51%)
val: 77090 транзакцій, фрод 2645 (3.43%)
test: 82325 транзакцій, фрод 2868 (3.48%)
xgboost 3.4.1 {'tree_method': 'hist', 'device': 'cuda'}


Підготовка

In [2]:
base = df.loc[m_tr, ["TransactionDT", "isFraud", "card1", "addr1",
                     "P_emaildomain", "DeviceInfo"]].copy()
base = base.sort_values("TransactionDT").reset_index(drop=True)
print("train:", len(base), "транзакцій")

_cache = {}
def past_stats(cols, delay_days):
    """Для кожної транзакції: скільки ранніших у групі (n_prior),
    скільки з них із відомою на цей момент міткою (n_known), скільки з них фрод."""
    key = (tuple(cols), delay_days)
    if key in _cache: return _cache[key]
    d = base.dropna(subset=cols)
    t, f = d["TransactionDT"].values, d["isFraud"].values
    n_prior = np.zeros(len(d), dtype=np.int32)
    n_known = np.zeros(len(d), dtype=np.int32)
    fr_known = np.zeros(len(d), dtype=np.int32)
    for pos in d.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        tt, ff = t[pos], f[pos]
        cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")            # ранніші в групі
        k = np.minimum(np.searchsorted(tt, tt - delay_days * 86400, side="right"), a)
        n_prior[pos], n_known[pos], fr_known[pos] = a, k, cs[k]
    out = d.assign(n_prior=n_prior, n_known=n_known, fr_known=fr_known)
    _cache[key] = out
    return out

keys = [["card1"], ["addr1"], ["P_emaildomain"], ["DeviceInfo"],
        ["card1", "addr1"], ["card1", "addr1", "P_emaildomain"]]

train: 431125 транзакцій


Чи є сигнал «лише по минулому»

In [3]:
rows = []
for cols in keys:
    for dl in (0, 7, 30):
        s = past_stats(cols, dl)
        has = s.n_known > 0
        r = (s.fr_known / s.n_known.where(has))
        y = s.isFraud
        a, b = r[has & (y == 1)].mean(), r[has & (y == 0)].mean()
        rows.append({"ключ": "+".join(cols), "затримка міток, дн": dl,
                     "охоплення %": round(has.sum() / len(base) * 100, 1),
                     "частка фроду серед відомих сусідів: у фрод-вузла %": round(a * 100, 1),
                     "у чесного %": round(b * 100, 1),
                     "відношення": round(a / b, 1) if b > 0 else None})
past_tbl = pd.DataFrame(rows)
display(past_tbl)
past_tbl.to_csv("results/ieee_cis_key_signal_past_only.csv", index=False)

,ключ,"затримка міток, дн",охоплення %,частка фроду серед відомих сусідів: у фрод-вузла %,у чесного %,відношення
0,card1,0,97.1,13.5,2.6,5.2
1,card1,7,90.0,9.9,2.6,3.7
2,card1,30,66.1,8.5,2.5,3.4
3,addr1,0,88.5,3.1,2.2,1.5
4,addr1,7,82.2,2.5,2.1,1.2
5,addr1,30,60.5,2.3,2.1,1.1
6,P_emaildomain,0,84.5,3.9,3.1,1.3
7,P_emaildomain,7,79.3,3.8,3.1,1.2
8,P_emaildomain,30,57.9,3.7,3.1,1.2
9,DeviceInfo,0,21.4,20.6,3.8,5.5


Розміри груп і поріг хабів

In [4]:
rows = []
for cols in keys:
    s = past_stats(cols, 0)
    p = s.n_prior
    rows.append({"ключ": "+".join(cols),
                 "охоплення %": round(len(s) / len(base) * 100, 1),
                 "медіана": float(p.median()), "p90": float(p.quantile(0.9)),
                 "p99": float(p.quantile(0.99)), "максимум": int(p.max()),
                 **{f"раніших >{c}, %": round((p > c).mean() * 100, 1) for c in (20, 50, 100, 500)}})
deg_tbl = pd.DataFrame(rows)
display(deg_tbl)
deg_tbl.to_csv("results/ieee_cis_degree_at_arrival.csv", index=False)

,ключ,охоплення %,медіана,p90,p99,максимум,"раніших >20, %","раніших >50, %","раніших >100, %","раніших >500, %"
0,card1,100.0,227.0,2740.0,8546.76,11298,79.0,69.8,61.5,37.3
1,addr1,88.6,5216.0,21572.0,30658.00,34185,99.4,98.8,97.9,92.1
2,P_emaildomain,84.5,30689.0,129973.6,162746.56,166388,99.7,99.2,98.5,95.3
3,DeviceInfo,21.7,5901.0,28149.6,36575.76,37512,88.2,83.4,80.0,74.7
4,card1+addr1,88.6,28.0,501.0,2584.00,4260,55.0,40.7,30.3,10.0
5,card1+addr1+P_emaildomain,73.5,6.0,123.0,753.00,1608,30.9,18.9,11.8,2.1


In [5]:
import os
os.makedirs("data/processed", exist_ok=True)     # у git не йде (.gitignore)

N = len(df)
ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))                    # час, при рівності ID
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))   # 0 train, 1 val, 2 test

def build_edges(cols, K, W_days):
    valid = df[cols].notna().all(axis=1).values
    idx = np.flatnonzero(valid)
    sub = df.loc[valid, cols]
    S, D = [], []
    for pos in sub.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]
        rows = rows[np.argsort(rank[rows], kind="stable")]       # за часом
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K,
                        np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo                                   # скільки попередніх беремо
        tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    if not S: return np.empty(0, np.int32), np.empty(0, np.int32)
    return np.concatenate(S), np.concatenate(D)

REL = {"u3": (["card1", "addr1", "P_emaildomain"], 50, 30),
       "u2": (["card1", "addr1"],                  50, 30),
       "c1": (["card1"],                           20, 30)}

edges = {}
for name, (cols, K, W) in REL.items():
    t0 = time.time()
    edges[name] = build_edges(cols, K, W)
    print(f"{name}: ребер {len(edges[name][0]):,} | {time.time() - t0:.0f} с")

# --- перевірки на витік ---
for name, (cols, K, W) in REL.items():
    s, d = edges[name]
    assert (rank[s] < rank[d]).all(), f"{name}: ребро не з минулого в майбутнє"
    assert (ts[s] <= ts[d]).all()
    assert (s != d).all()
    assert len(np.unique(s.astype(np.int64) * N + d)) == len(s), f"{name}: дублікати ребер"
    assert np.bincount(d, minlength=N).max() <= K, f"{name}: перевищено K"
    assert (split_of[s[split_of[d] == 0]] == 0).all(), f"{name}: train-вузол отримує з val/test"
    smp = np.random.default_rng(0).choice(len(s), min(200_000, len(s)), replace=False)
    assert (df[cols].values[s[smp]] == df[cols].values[d[smp]]).all(), f"{name}: ключ не збігається"
print("усі перевірки пройдено")

# --- статистика ---
rows = []
for name, (s, d) in edges.items():
    indeg = np.bincount(d, minlength=N)
    for sp, lab in [(0, "train"), (1, "val"), (2, "test")]:
        m = split_of == sp
        rows.append({"відношення": name, "частина": lab, "ребер": int(m[d].sum()),
                     "вузлів з ≥1 вхідним ребром, %": round((indeg[m] > 0).mean() * 100, 1),
                     "середній вхідний ступінь": round(indeg[m].mean(), 2)})
edge_tbl = pd.DataFrame(rows)
display(edge_tbl)
edge_tbl.to_csv("results/ieee_cis_graph_v1_stats.csv", index=False)

wk_deg = pd.DataFrame({n: pd.Series(np.bincount(e[1], minlength=N)).groupby(df.week.values).mean().round(1)
                       for n, e in edges.items()})
display(wk_deg)

for name, (s, d) in edges.items():
    np.savez_compressed(f"data/processed/graph_v1_{name}.npz",
                        src=s, dst=d, dt=(ts[d] - ts[s]).astype(np.int32))

u3: ребер 5,295,877 | 1 с
u2: ребер 12,260,904 | 1 с
c1: ребер 9,739,081 | 0 с
усі перевірки пройдено


,відношення,частина,ребер,"вузлів з ≥1 вхідним ребром, %",середній вхідний ступінь
0,u3,train,3789213,55.5,8.79
1,u3,val,748184,59.0,9.71
2,u3,test,758480,57.0,9.21
3,u2,train,8753748,78.7,20.30
4,u2,val,1690298,82.6,21.93
5,u2,test,1816858,81.8,22.07
6,c1,train,7058739,96.1,16.37
7,c1,val,1296620,97.2,16.82
8,c1,test,1383722,97.0,16.81


,u3,u2,c1
0,2.2,8.4,11.6
1,4.8,14.4,14.8
2,6.4,17.3,15.8
3,8.6,20.1,16.6
4,10.1,23.2,17.3
5,10.1,23.5,17.1
6,10.3,23.5,17.0
7,9.8,22.5,16.8
8,8.8,20.7,16.8
9,10.0,22.6,16.9


XGBoost з ознаками сусідів

In [6]:
agg_cols = ["TransactionAmt"] + [f"C{i}" for i in range(1, 15)]
amt = df["TransactionAmt"].values.astype("float64")
out = {}
t0 = time.time()
for name, (s, d) in edges.items():
    dt = (ts[d] - ts[s]).astype("float64")
    out[f"{name}_deg"] = np.bincount(d, minlength=N).astype("float32")
    out[f"{name}_deg_24h"] = np.bincount(d[dt <= 86400], minlength=N).astype("float32")
    g = pd.DataFrame({"dt": dt, "d": d}).groupby("d")["dt"].agg(["min", "mean"])
    for stat in ("min", "mean"):
        a = np.full(N, np.nan, dtype="float32")
        a[g.index.values] = g[stat].values
        out[f"{name}_dt_{stat}"] = a
    for c in agg_cols:
        v = df[c].values.astype("float64")[s]
        ok = ~np.isnan(v)
        ssum = np.bincount(d[ok], weights=v[ok], minlength=N)
        scnt = np.bincount(d[ok], minlength=N)
        out[f"{name}_mean_{c}"] = np.where(scnt > 0, ssum / np.maximum(scnt, 1), np.nan).astype("float32")
    with np.errstate(divide="ignore", invalid="ignore"):
        r = amt / out[f"{name}_mean_TransactionAmt"]
    out[f"{name}_amt_ratio"] = np.where(np.isfinite(r), r, np.nan).astype("float32")
    print(name, "готово,", round(time.time() - t0), "с")

G = pd.DataFrame(out)
deg_names = {"deg", "deg_24h", "dt_min", "dt_mean"}
deg_cols = [c for c in G.columns if c[3:] in deg_names]
agg_feat_cols = [c for c in G.columns if c not in deg_cols]
print("deg-ознак:", len(deg_cols), "| agg-ознак:", len(agg_feat_cols))

u3 готово, 2 с
u2 готово, 7 с
c1 готово, 11 с
deg-ознак: 12 | agg-ознак: 48


Два варіанти, спершу по одному seed

In [8]:
y = df.isFraud.values
assert isinstance(y, np.ndarray) and len(y) == N and y.sum() == 20663
print("y відновлено:", y.shape, "| фрод:", int(y.sum()))

y відновлено: (590540,) | фрод: 20663


In [9]:
def run_xgb_X(Xf, seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                      eval_metric="aucpr", early_stopping_rounds=50,
                      random_state=seed, **DEV)
    m.fit(Xf[m_tr], y[m_tr], eval_set=[(Xf[m_va], y[m_va])], verbose=False)
    r = metrics(y[m_te], m.predict_proba(Xf[m_te])[:, 1])
    r.update(seed=seed, best_iter=m.best_iteration, val_aucpr=m.best_score)
    return r, m

variants = {"base": [], "+deg": deg_cols, "+deg+agg": deg_cols + agg_feat_cols}
XV = {k: pd.concat([X, G[c]], axis=1) if c else X for k, c in variants.items()}

for k in ("+deg", "+deg+agg"):
    t0 = time.time()
    r, mdl = run_xgb_X(XV[k], 0)
    print(k, {a: round(float(b), 4) for a, b in r.items() if a in ("pr_auc", "roc_auc", "recall@k", "precision@1%")},
          "| дерев", r["best_iter"], "| секунд", round(time.time() - t0))

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [16:11:02] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


+deg {'pr_auc': 0.5001, 'roc_auc': 0.8875, 'recall@k': 0.4819, 'precision@1%': 0.87} | дерев 630 | секунд 37
+deg+agg {'pr_auc': 0.505, 'roc_auc': 0.8864, 'recall@k': 0.4819, 'precision@1%': 0.8748} | дерев 542 | секунд 39


In [10]:
res = []
for k in ["base", "+deg", "+deg+agg"]:
    for sd in range(5):
        r, _ = run_xgb_X(XV[k], sd)
        r["variant"] = k
        res.append(r)
        print(k, sd, round(float(r["pr_auc"]), 4), flush=True)
res1 = pd.DataFrame(res)
res1.to_csv("results/ieee_cis_xgb_graphfeat.csv", index=False)
res1.groupby("variant")[["pr_auc", "roc_auc", "recall@k", "precision@1%"]].agg(["mean", "std"]).round(3)

base 0 0.4998
base 1 0.5064
base 2 0.5059
base 3 0.5064
base 4 0.5068
+deg 0 0.5001
+deg 1 0.5098
+deg 2 0.5063
+deg 3 0.5061
+deg 4 0.5076
+deg+agg 0 0.505
+deg+agg 1 0.5003
+deg+agg 2 0.5059
+deg+agg 3 0.4967
+deg+agg 4 0.4987


pr_auc        roc_auc        recall@k        precision@1%       
           mean    std    mean    std     mean    std         mean    std
variant                                                                  
+deg      0.506  0.004   0.892  0.004    0.488  0.004        0.873  0.004
+deg+agg  0.501  0.004   0.887  0.006    0.482  0.005        0.872  0.004
base      0.505  0.003   0.886  0.005    0.487  0.006        0.876  0.007

Важливість ознак

In [11]:
r, mdl = run_xgb_X(XV["+deg+agg"], 0)
imp = pd.Series(mdl.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
display(imp.head(20).round(1))
imp.head(40).round(2).to_csv("results/ieee_cis_xgb_importance.csv")

V258     8455.2
V257     3718.2
V70      3554.1
V294     2103.9
V201     1590.1
V91      1117.4
V327      932.5
C14       888.4
card6     722.7
V308      669.7
C8        662.7
V69       653.4
C5        618.9
V317      550.5
V98       468.9
V187      445.6
V141      421.2
C10       410.3
V30       351.3
V177      326.4
dtype: float64

Варіанти без V

In [13]:
import re

vcols = [c for c in X.columns if re.fullmatch(r"V\d+", c)]
print("V-колонок:", len(vcols))
XnoV = X.drop(columns=vcols)
variants_noV = {"noV": XnoV,
                "noV+deg+agg": pd.concat([XnoV, G[deg_cols + agg_feat_cols]], axis=1)}
res = []
for k, Xf in variants_noV.items():
    for sd in range(5):
        r, _ = run_xgb_X(Xf, sd)
        r["variant"] = k
        res.append(r)
        print(k, sd, round(float(r["pr_auc"]), 4), flush=True)
res2 = pd.DataFrame(res)
res2.to_csv("results/ieee_cis_xgb_graphfeat_noV.csv", index=False)
res2.groupby("variant")[["pr_auc", "roc_auc", "recall@k", "precision@1%"]].agg(["mean", "std"]).round(3)

V-колонок: 339
noV 0 0.4803
noV 1 0.4917
noV 2 0.4877
noV 3 0.4832
noV 4 0.4833
noV+deg+agg 0 0.4827
noV+deg+agg 1 0.4734
noV+deg+agg 2 0.4775
noV+deg+agg 3 0.4805
noV+deg+agg 4 0.4721


pr_auc        roc_auc        recall@k        precision@1%       
              mean    std    mean    std     mean    std         mean    std
variant                                                                     
noV          0.485  0.004   0.891  0.002    0.474  0.003        0.842  0.004
noV+deg+agg  0.477  0.005   0.886  0.003    0.466  0.003        0.840  0.009